In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
%pip install -qU "langchain[groq]" langchain-huggingface langchain-chroma langchain
%pip install --upgrade "langchain[all]"

%pip install -qqq html2text==2024.2.26 --progress-bar off
%pip install -q unstructured playwright
!pip install tensorflow -q
!playwright install

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.2/145.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 21.6 MB/s eta 0

In [ ]:
# Colab-specific
from google.colab import userdata
from google.colab import files
import zipfile

# System utilities
import os
import getpass
import random
import string
from operator import add

# Web scraping and parsing
import bs4
from bs4 import BeautifulSoup
import html2text
from tqdm import tqdm
from playwright.async_api import async_playwright

# Typing
from typing import List, Sequence, Literal, Annotated
from typing_extensions import TypedDict

# LangChain Core
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, RemoveMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

# LangChain + Integrations
from langchain.chat_models import init_chat_model
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LangGraph
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver, MemorySaver
from langgraph.graph.message import add_messages

# TensorFlow + Model Utilities + Image Processing
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import Metric
from tensorflow.keras.losses import Loss
from tensorflow.keras import backend as K
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image
import kagglehub
from PIL import Image
from io import BytesIO

In [ ]:
# @title CNN
class BalancedAccuracy(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name='balanced_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.confusion_matrix = self.add_weight(
            name='confusion_matrix',
            shape=(num_classes, num_classes),
            initializer='zeros'
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.argmax(y_pred, axis=1, output_type=tf.int32)
        batch_cm = tf.math.confusion_matrix(y_true, y_pred,
                                           num_classes=self.num_classes)
        self.confusion_matrix.assign_add(batch_cm)

    def result(self):
        eps = tf.keras.backend.epsilon()
        tp = tf.linalg.diag_part(self.confusion_matrix)
        class_counts = tf.reduce_sum(self.confusion_matrix, axis=1)
        recall_per_class = tp / (class_counts + eps)
        return tf.reduce_mean(recall_per_class)

    def reset_states(self):
        self.confusion_matrix.assign(tf.zeros_like(self.confusion_matrix))

    def get_config(self):
        config = super().get_config()
        config.update({'num_classes': self.num_classes})
        return config

class WeightedF1(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name='weighted_f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.cm = self.add_weight(
            name='confusion_matrix',
            shape=(num_classes, num_classes),
            initializer='zeros'
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.argmax(y_pred, axis=1, output_type=tf.int32)
        batch_cm = tf.math.confusion_matrix(y_true, y_pred,
                                           num_classes=self.num_classes)
        self.cm.assign_add(batch_cm)

    def result(self):
        eps = tf.keras.backend.epsilon()
        tp = tf.linalg.diag_part(self.cm)
        pred_sums = tf.reduce_sum(self.cm, axis=0)
        true_sums = tf.reduce_sum(self.cm, axis=1)

        precision = tp / (pred_sums + eps)
        recall = tp / (true_sums + eps)
        f1 = (2 * precision * recall) / (precision + recall + eps)

        weights = true_sums / (tf.reduce_sum(true_sums) + eps)
        return tf.reduce_sum(f1 * weights)

    def reset_states(self):
        self.cm.assign(tf.zeros_like(self.cm))

    def get_config(self):
        config = super().get_config()
        config.update({'num_classes': self.num_classes})
        return config


class WeightedLoss(tf.keras.losses.Loss):
    def __init__(self, class_weights, name="weighted_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.class_weights = tf.constant(class_weights, dtype=tf.float32)

    def get_config(self):
        config = super().get_config()
        config.update({"class_weights": self.class_weights.numpy().tolist()})
        return config

    def call(self, y_true, y_pred):
        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        sample_weights = tf.gather(self.class_weights, tf.cast(y_true, tf.int32))
        return tf.reduce_mean(ce * sample_weights)


target_size = (380, 380);
class_names= ['Acne', 'Actinic keratosis', 'Atopic Dermatitis', 'Basal Cell Carcinoma', 'Benign keratosis', 'Bullous', 'Candidiasis', 'Dermatofibroma','DrugEruption',
              'Eczema', 'Infestations_Bites', 'Lupus', 'Melanocytic Nevi', 'Melanoma', 'MonkeyPox', 'Psoriasis Lichen Planus', 'Rosacea', 'Seborrheic Keratoses',
              'SkinCancer', 'Squamous cell carcinoma', 'Sunlight Damage', 'Tinea Ringworm Candidiasis', 'Vascular lesion', 'Vascular_Tumors', 'Vasculitis', 'Warts Molluscum']


loaded_model = tf.keras.models.load_model(
    '/content/effnetb4.keras',
    custom_objects={
        'BalancedAccuracy': BalancedAccuracy,
        'WeightedF1': WeightedF1,
        'WeightedLoss': WeightedLoss
    }
)

def predict_class(image_file):
    img_bytes = BytesIO(image_file.read())  # Convert to BytesIO
    img = image.load_img(img_bytes, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    prediction = loaded_model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction, axis=1)[0]]

    return predicted_class

In [ ]:
#@title LLM
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
if not os.environ.get("LANGSMITH_API_KEY"):
  os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter API key for LANGSMITH: ")

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

In [ ]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

llm = init_chat_model("llama3-70b-8192", model_provider="groq", rate_limiter= rate_limiter)
#llama3-8b-8192

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
#@title Execeute if you have a zipped db
uploaded = files.upload()

for zip_filename in uploaded.keys():
    folder_name = os.path.splitext(zip_filename)[0]
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(folder_name)

In [ ]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db1",  # Where to save data locally, remove if not necessary
)

markdown_converter = html2text.HTML2Text()
markdown_converter.ignore_links = False
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)

USER_AGENT = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"

async def fetch_page(url, user_agent=USER_AGENT) -> str:
    # Launch browser and navigate to the URL
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch()
        context = await browser.new_context(user_agent=user_agent)
        page = await context.new_page()
        await page.goto(url)
        content = await page.content()
        current_url = page.url

        await browser.close()

    # Convert HTML to Markdown
    markdown_content = markdown_converter.handle(content)

    soup = BeautifulSoup(content, "html.parser")
    title = soup.title.string.strip() if soup.title else "No title"
    description = soup.find("meta", attrs={"name": "description"})
    meta_description = description["content"].strip() if description else "No description"
    metadata = {
            "source": current_url,
            "title": title,
            "description": meta_description,
    }
    return markdown_content, metadata

df = pd.read_csv('diseases.csv', header = None)
urls = df.iloc[:, 0].tolist()

for url in tqdm(urls):
    markdown_content, metadata = await fetch_page(url)

    doc = [Document(page_content=markdown_content, metadata = metadata)]

    all_splits = text_splitter.split_documents(doc)
    _ = vector_store.add_documents(documents=all_splits)

In [ ]:
class State(MessagesState):
    summary: str
    #context: List[Document]
    language: str
    dis: str

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )

    return serialized, retrieved_docs

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system",
         '''You are an assistant for question-answering tasks.
         Your name is 'NeuraSkin Bot' and is specialised in detecting Human skin diseases
        The user is diagonised with the skin disease: {dis}.
        Answer all questions to the best of your ability in {language}.
        Use three to four sentences maximum and keep the answer concise.
        If the answer to query is not found in the retrieved context, just say that you don't know, don't try to make up an answer.
        End with a question making the user to ask another question.
        Use the following pieces of retrieved context to answer the question.
        HERE THE CONTENT STARTS:
        {context}
        HERE THE CONTENT ENDS:

        {history}
        \n\n''',),

        MessagesPlaceholder(variable_name="messages"),
    ]
)

def query_or_respond(state: State):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve], tool_choice="auto")

    summary = state.get("summary", "")
    if summary:
        system_message = f"Summary of conversation earlier: {summary} Current Query: "
        messages = [SystemMessage(content=system_message)] + state["messages"]
    else:
        messages = state["messages"]


    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}

tools = ToolNode([retrieve])

def generate(state: State):
    """Generate answer."""
    recent_tool_messages = []

    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    message_history = []

    for message in state["messages"]:
        if message.type == "human":
            message_history.append(HumanMessage(message.content))
        elif message.type == "ai" and not message.tool_calls:
            message_history.append(AIMessage(message.content))

    summary = state.get("summary", "")

    if summary:
        system_message = f"Summary of conversation earlier: {summary}"
        messages = [SystemMessage(content=system_message)] + message_history
    else:
        messages = message_history

    response = llm.invoke(messages)

    docs_content = "\n\n".join(doc.content for doc in tool_messages)

    prompt = prompt_template.invoke({"language": state["language"],
                                     "messages": message_history,
                                     "history": summary,
                                     "context": docs_content,
                                     "dis": state["dis"]})

    response = llm.invoke(prompt)
    return {"messages": [response]}


def summarize_conversation(state: State):
    message_history = []
    last_message = []
    for message in state["messages"]:
        if message.type == "human":
            message_history.append(HumanMessage(message.content))
            last_message.append(message)
        elif message.type == "ai" and not message.tool_calls:
            message_history.append(AIMessage(message.content))
            last_message.append(message)

    if len(message_history) > 6:
        # First, we summarize the conversation
        summary = state.get("summary", "")
        template = '''Create a short and concise summary (include the users message and reduce the ai messages to a maximum extent) for the conversation above:
            for example the message:
            user: "I am XYZW of age 10"
            ai: <ai message>

            user: "what are the symtoms of mpox"
            ai message: "mpox causes signs and symptoms that usually begin within a week but can start 1-21 days after exposure. Common symptoms of mpox include:
            * Rash * Fever * Sore throat ..."

            should be reduced to
            user: revealed his name to be XYZW of age 10
            user: mention the jist of the users question
            ai: "ai replied about the symtoms of mpox"

            name of the user: XYZW (if provided)
            age of the user: 10 (if provided)
            '''

        if summary:
            # If a summary already exists, we use a different system prompt to summarize it than if one didn't
            summary_message = (
                "Extend the below old summary by taking into account the new messages above: (keep the details of user from the past summary if provided)"
                f"This is summary of the conversation to date: {summary}\n\n"

                f"{template}"
            )
        else:
            summary_message = template

        messages = message_history + [HumanMessage(content=summary_message)]
        response = llm.invoke(messages)
        print(last_message[-1:-3:-1])
        delete_messages = [RemoveMessage(id=m.id) for m in state["messages"] if m not in last_message[-1:-3:-1]]
        print(delete_messages)
        return {"summary": response.content, "messages": delete_messages}


In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node("generate", generate)
graph_builder.add_node(summarize_conversation)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: "summarize_conversation", "tools": "tools"},
)
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", "summarize_conversation")
graph_builder.add_edge("summarize_conversation", END)

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
def get_unique_id(length=3):
    chars = string.ascii_letters + string.digits
    return ''.join(random.choices(chars, k=length))

config = {"configurable": {"thread_id": get_unique_id()}}

set_llm_cache(InMemoryCache())

In [ ]:
#@title BACKEND FRONTEND CONNECTION
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!rm cloudflared-linux-amd64.deb
!pip install flask flask_cors -q

--2025-04-20 09:39:15--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2025.4.0/cloudflared-linux-amd64.deb [following]
--2025-04-20 09:39:15--  https://github.com/cloudflare/cloudflared/releases/download/2025.4.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://objects.githubusercontent.com/github-production-release-asset-2e65be/106867604/d7e7703c-c0be-4512-b40f-145c402e03fd?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=releaseassetproduction%2F20250420%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20250420T093915Z&X-Amz-Expires=300&X-Amz-Signature=2da01d35360456395fe84e66ba89b1926038dffac13a43688faf6bbb4afa

In [ ]:
!pkill -f flask

In [ ]:
!fuser -k 8000/tcp

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

@app.route('/chat', methods=['POST'])
def chat():
    user_message = request.form.get("message", "")
    image_file = request.files.get("image")
    language = "English"

    bot_reply = "Sorry, I couldn't understand that."

    if image_file:
        img_bytes = BytesIO(image_file.read())  # Convert to BytesIO
        img = image.load_img(img_bytes, target_size=target_size)
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)
        prediction = loaded_model.predict(img_array)
        predicted_class = class_names[np.argmax(prediction, axis=1)[0]]

        user_message = f"The user is diagonised with {predicted_class}, give the responce for user's question: {user_message}"

        for step in graph.stream(
            {"messages": [{"role": "user", "content": user_message}], "language": language, "dis": predicted_class},
            stream_mode="values",
            config = config
        ):

            response = step["messages"][-1]
            if response.type == "ai" and not response.tool_calls:
                    bot_reply = response.content

    else:
        for step in graph.stream(
            {"messages": [{"role": "user", "content": user_message}], "language": language, "dis": "Diagnosis not done yet"},
            stream_mode="values",
            config = config
        ):

            response = step["messages"][-1]
            if response.type == "ai" and not response.tool_calls:
                    bot_reply = response.content

    return jsonify({"response": bot_reply})

config["configurable"]["thread_id"] = get_unique_id()

@app.route('/run-backend-func', methods=['POST'])
def run_func():
    config["configurable"]["thread_id"] = get_unique_id()
    return "Backend function executed"

def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False) # Disable debug mode for production-like environment

if __name__ == '__main__':
    flask_thread = threading.Thread(target=run_app)
    flask_thread.daemon = True
    flask_thread.start()
    print("Flask app is running in the background on http://0.0.0.0:8001")



Flask app is running in the background on http://0.0.0.0:8001


In [ ]:
!cloudflared tunnel --url http://localhost:5000 Avi

2025-04-20T09:52:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-04-20T09:52:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-04-20T09:52:50Z INF +--------------------------------------------------------------------------------------------+
2025-04-20T09:52:50Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-04-20T09:52:50Z INF |  https://arena-lisa-expertise-ferry.trycloudflare.com 

INFO:werkzeug:127.0.0.1 - - [20/Apr/2025 09:53:27] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Apr/2025 09:55:08] "POST /chat HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


INFO:werkzeug:127.0.0.1 - - [20/Apr/2025 09:55:40] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Apr/2025 09:55:52] "POST /chat HTTP/1.1" 200 -


[AIMessage(content='As you have Acne, some common symptoms include blackheads, whiteheads, pimples, cysts, and nodules on the skin. These can appear on the face, neck, chest, back, or other areas of the body. You may also experience redness, inflammation, and skin irritation.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 842, 'total_tokens': 905, 'completion_time': 0.225791346, 'prompt_time': 0.030040803, 'queue_time': 0.209554869, 'total_time': 0.255832149}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_dd4ae1c591', 'finish_reason': 'stop', 'logprobs': None}, id='run-3ad1bf87-536e-420e-b02c-866dfbc4b21a-0', usage_metadata={'input_tokens': 842, 'output_tokens': 63, 'total_tokens': 905}), HumanMessage(content='what are symptoms of it', additional_kwargs={}, response_metadata={}, id='8569229f-854b-4e04-80ce-eb7fb9d124bf')]
[RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='057f2345-cc84-40f2-b419-